In [93]:
using Combinatorics, ClusteredLowRankSolver, Infinity, LinearAlgebra, Nemo
include("fun.jl")

# setprecision(10^3)

LDL (generic function with 1 method)

In [94]:
# [7,1,4], [19,1,8], [13,1,6]

n,K,d = [13,1,6]

R, x = polynomial_ring(QQ, :x)
N, z = number_field(x^2 - 3, :z)

name_file=string("cerf/$n","_$K","_$d.jl")

include(name_file) # includes Y-> set of positive matrices

56-element Vector{Matrix{Any}}:
 [8620373//125000*z + 11849483081//1000000 0 … -1534327//500000*z - 1055786049//1000000 -531427819//500000*z - 1151131//125000; 0 -4593699//200000*z + 321324591997//1000000 … 0 0; … ; -1534327//500000*z - 1055786049//1000000 0 … -3//100000*z + 201477201//1000000 -1//1000000*z + 95806201//250000; -531427819//500000*z - 1151131//125000 0 … -1//1000000*z + 95806201//250000 -117//1000000*z + 1318002609//1000000]
 [66167//200000*z + 192063749579//1000000 0 … 0 0; 0 66167//200000*z + 192063749579//1000000 … 0 0; … ; 0 0 … 39//250000*z + 141504323//500000 -23432039//100000; 0 0 … -23432039//100000 -219//1000000*z + 399455637//200000]
 [-3773//250000*z + 165512921141//1000000 0 … 0 0; 0 32973//1000000*z + 163026833771//1000000 … 0 0; … ; 0 0 … 9//1000000*z + 2333605781//1000000 5135559//3125; 0 0 … 5135559//3125 567//500000*z + 778614339//50000]
 [-2553//125000*z + 9203525607//62500 0 … 0 0; 0 -773917//882000000*z + 1781060665861//12250000 … 0 0; … ; 0 0 … 1751/

In [ ]:
f4, size = f4_fun(n)
f2, size2 = f2_fun(n)

y_field=yijtp_fun_field(Y, n, f2, z)

w_field=(-2*y_field[f4[n, 0, 0, 0]+1]-y_field[f4[n, n, n, n]+1])/gamma_field(n,0,0,0,n)

# Obj. funct

obj_fun = (2^n-1)*w_field-y_field[f4[0, 0, 0, 0]+1]

println("Objective function = $obj_fun")
println("\n")

# Constraint 1

ref=0

for i in d:n-1
    max=(gamma_field(i,i,i,i,n)*w_field+(2*y_field[f4[i, 0, 0, 0]+1]+y_field[f4[i, i, i, i]+1]))
    max=BigFloat(coeff(max,0))+BigFloat(coeff(max,1))*sqrt(big(3))
    max=abs(max)
    
    if max > ref
        ref=max
    end
end

println("Min value of the constraint 1 = $ref")
println("\n")

# Constraint 2

ref=0 

for (i, j, t, p) in keys(f4)
    if ((t - p) % 2 == 0 && i !=0 && j != 0 && i+j-t-p >= d)

        max=y_field[f4[i,j,t,p]+1]

        max=BigFloat(coeff(max,0))+BigFloat(coeff(max,1))*sqrt(big(3))
        
        max=abs(max)
        
        if max > ref
            ref=max
        end
    end
end

println("Min value of the constraint 2 = $ref")

Objective function = 1


Min value of the constraint 1 = 0


Min value of the constraint 2 = 0


In [96]:
# Maximum and minimum eigenvalue of Y

Ypsd=[]

for k in 1:size2
    dim=length(Y[k][1,:])
    
    M=Matrix(undef,dim,dim)
    
    for i in 1:dim
        for j in 1:dim
            
            if typeof(Y[k][i,j])==AbsSimpleNumFieldElem
                
                M[i,j]=BigFloat(coeff(Y[k][i,j],0))+BigFloat(coeff(Y[k][i,j],1))*sqrt(big(3))
            else
                M[i,j]=Y[k][i,j]
               
        
            end
        end
    end
    
    push!(Ypsd,M)
    
end

ref_min=Inf
ref_max=0
for i in 1:size2
    mat=Matrix{BigFloat}(Ypsd[i])
    
    eigen=eigvals(mat)

    eigen_abs=abs.(eigen)

    min=minimum(eigen_abs)
    max=maximum(eigen_abs)

    if ref_min > min
        ref_min=min
    end
    if max > ref_max
        ref_max=max
    end
end

println("Minimum eigenvalue = $ref_min\n")
println("Maximum eigenvalue = $ref_max")

Minimum eigenvalue = 0.000307168998923665020929026237468667664182515635384864766537911861062045883410669081646304308378958424945084921978767073052507269430859113503266079923039640340951517111370192422043128814934214098124792763454690766683412077718859437646021361435316156889124673599456961697703905649853586589317052014676876248

Maximum eigenvalue = 898940.318431183907176448332783890588734230171141005463785484046661296060348193318063941359476990883591352777994109511499660483654642561549474735952003283019287593872271722565112775930386137254196517315781545491955537432651426989551288010913992104102544166286688725765612796770046255690924818934872949367


In [ ]:
# Check positivity with LDL method

ref_min=Inf

upper=rationalize(1.73206)
lower=rationalize(1.73205) 

for k in 1:size2
    A=Y[k]
    D=length(A[1,:])
    d=LDL(A,D)
    d_factor=Matrix(undef,D,1)

    for j in 1:D

        if coeff(d[j],1) < 0
            factor=coeff(d[j],0)+upper*coeff(d[j],1) # bigger value of sqrt(3)
        else
            factor=coeff(d[j],0)+lower*coeff(d[j],1) # smaller value of sqrt(3)
        end
        
        d_factor[j]=factor
    end
    
    ref=minimum(d_factor)

    if ref_min > ref
        ref_min=ref
    end
end

ref_min_float=BigFloat(ref_min)

println("Minimum value (Fraction) = $ref_min")
println("Minimum value (Float) = $ref_min_float")

Minimum value (fraction) = 41401203306198434590340133023767460617222661761021905700609642028794503926355173//161021710675959866942057967850579157966288779332835701771598174014720000000000000
Minimum value (Float) = 0.257115659325680790024111905862768337331454793424200784399738646284850086972305135647805185613291456941469027045931402890206233146769990544512166570076265825300023891788557741851357933312267313024041616585003261041155972531666891483721858097495026266756670801855284332036560779912836149571249604954202717
